# SS3 Part B — STADIOEquities customer activation prediction

**Final chosen method:** Random Forest classifier with a leak-free preprocessing pipeline.  
**Target:** `activation_outcome` — whether a client activated by making their first successful deposit, as labelled by the client.  
**Unit of analysis:** one registered client. **Prediction time:** at registration.  

The supplied archive also contains a separate dormancy target. This notebook intentionally trains **one** model (activation), because the two targets have different eligible populations and permissible predictors. All random seeds are fixed to 42.

> The source README describes these records as **synthetic** and warns that many current-state and post-deposit fields leak the activation label. This notebook uses only fields available when a client registered.

## 1. Dataset loading (2 marks)

Load the two necessary CSV tables directly from  `data/table1_client_account.csv` (registration-time predictors) and `table9_outcome_target.csv` (labels only). The original files each contain **3,109 rows**, but **nine duplicate client rows** are removed, leaving **3,100 unique clients**. The binary classes are **activated (True, 1)** and **not activated (False, 0)**. The following code prints actual counts, proportions, and the majority-to-minority imbalance ratio rather than relying on manually entered estimates.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score,
                             roc_curve, precision_recall_curve, average_precision_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
import joblib

RANDOM_STATE = 42
def read_from_archive(filename):
    file = 'data/' + filename
    return pd.read_csv(file)

accounts_raw = read_from_archive('table1_client_account.csv')
labels_raw = read_from_archive('table9_outcome_target.csv')
print('Raw account rows:', len(accounts_raw))
print('Raw outcome rows:', len(labels_raw))
for name, data in [('accounts', accounts_raw), ('outcomes', labels_raw)]:
    conflicts = data.groupby('client_id').size().gt(1).sum()
    print(f'{name}: {conflicts} client IDs repeated')

assert labels_raw.groupby('client_id')['activation_outcome'].nunique().max() == 1, 'Conflicting target labels: investigate duplicates'
accounts = accounts_raw.drop_duplicates(subset='client_id', keep='first').copy()
labels = labels_raw.drop_duplicates(subset='client_id', keep='first').copy()
df = accounts.merge(labels[['client_id','activation_outcome']], on='client_id', how='inner', validate='one_to_one')
assert len(df) == df.client_id.nunique()
assert df.activation_outcome.notna().all()
y_all = df['activation_outcome'].astype(int)
counts = y_all.value_counts().reindex([0,1], fill_value=0)
print(f'Unique usable clients: {len(df):,}')
print(f'Not activated (0): {counts[0]:,} ({counts[0]/len(df):.1%})')
print(f'Activated (1): {counts[1]:,} ({counts[1]/len(df):.1%})')
print(f'Imbalance ratio (majority:minority): {counts.max()/counts.min():.2f}:1')
display(df[['client_id', 'registration_date', 'activation_outcome']].head())

Raw account rows: 3109
Raw outcome rows: 3109
accounts: 9 client IDs repeated
outcomes: 9 client IDs repeated
Unique usable clients: 3,100
Not activated (0): 1,567 (50.5%)
Activated (1): 1,533 (49.5%)
Imbalance ratio (majority:minority): 1.02:1


,client_id,registration_date,activation_outcome
0,C102938,2023-03-24 07:22:48,True
1,C102939,2023-03-23 07:01:16,False
2,C102943,2023-04-13 18:58:00,True
3,C102944,2022-11-08 22:39:28,False
4,C102946,2023-05-30 17:09:31,False


## 2. Dataset preprocessing and splitting

**Why these steps?**

- **Prevent target leakage:** account status, KYC completion, deposits, transactions, trades, activity flags, and all post-registration events would reveal information unavailable at registration. Only registration-time demographics, stated preferences, acquisition attributes and the calendar month are features; table 9 contributes **only the target**.
- **Data quality:** trim and standardise inconsistent categorical spellings, convert invalid ages (under 18 or above 100) and invalid negative acquisition costs to missing values, and deduplicate on anonymised `client_id`.
- **Missing values and categorical encoding:** fit median imputation to numeric fields and most-frequent imputation + one-hot encoding to categories. Fit these transformers **on the training set only**, preventing information from the validation/test sets influencing training. One-hot encoding avoids inventing arbitrary numerical ordering.
- **Chronological 60% / 20% / 20% split:** older signups train the model, later signups validate it, and newest signups test final generalisation. This better reflects deployment to future registrations than a purely random split. **No client appears in multiple subsets.** Validation is used to assess the chosen configuration; the test set is examined once for final reporting.
- **No feature scaling:** Random Forest splits are insensitive to feature scale; scaling is unnecessary. The model can be revisited with a future as-of-7-days design only if outcomes and event timestamps are censored consistently.

In [4]:
# Parse mixed client-supplied date formats; day-first for slash-formatted values.
parsed_dates = pd.to_datetime(df['registration_date'], format='mixed', dayfirst=True, errors='coerce')
if parsed_dates.isna().any():
    raise ValueError(f'{parsed_dates.isna().sum()} registration dates could not be parsed; examine before proceeding.')
df['registration_timestamp'] = parsed_dates

numeric_features = ['age', 'acquisition_cost', 'registration_month']
categorical_features = ['province_region', 'stated_investment_goal', 'risk_appetite',
                        'client_segment', 'acquisition_channel', 'acquisition_campaign']

for col in categorical_features:
    df[col] = df[col].astype('string').str.strip().str.lower()
    df[col] = df[col].replace({'': pd.NA, 'nan': pd.NA, 'none': pd.NA, 'unknown': pd.NA}).astype(object)
    df[col] = df[col].where(pd.notna(df[col]), np.nan)
for col in ['age','acquisition_cost']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df.loc[~df['age'].between(18,100), 'age'] = np.nan
df.loc[df['acquisition_cost'] < 0, 'acquisition_cost'] = np.nan
df['registration_month'] = df['registration_timestamp'].dt.month.astype(float)

# Sort by actual date, with stable client ID tie-breaker; keep every subset chronologically ordered.
df = df.sort_values(['registration_timestamp', 'client_id']).reset_index(drop=True)
n = len(df)
train_end, val_end = int(0.60*n), int(0.80*n)
train_df, val_df, test_df = df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]
X_train, X_val, X_test = (part[numeric_features + categorical_features] for part in (train_df,val_df,test_df))
y_train, y_val, y_test = (part['activation_outcome'].astype(int) for part in (train_df,val_df,test_df))
assert not (set(train_df.client_id) & set(val_df.client_id))
assert not (set(train_df.client_id) & set(test_df.client_id))
assert train_df.registration_timestamp.max() <= val_df.registration_timestamp.min()
assert val_df.registration_timestamp.max() <= test_df.registration_timestamp.min()
for name, part in [('Train',train_df),('Validation',val_df),('Test',test_df)]:
    print(f'{name:10} n={len(part):4} | Activated={part.activation_outcome.mean():.1%} | '
          f'{part.registration_timestamp.min().date()} to {part.registration_timestamp.max().date()}')
print('Missing numeric values in training before imputation:')
print(X_train[numeric_features].isna().sum())

Train      n=1860 | Activated=51.5% | 2022-09-01 to 2024-11-22
Validation n= 620 | Activated=46.9% | 2024-11-22 to 2025-08-19
Test       n= 620 | Activated=46.0% | 2025-08-20 to 2026-05-06
Missing numeric values in training before imputation:
age                   5
acquisition_cost      0
registration_month    0
dtype: int64


## 3. Model training

**Why Random Forest?** It is a strong, interpretable-feature-importance baseline for small-to-medium tabular datasets and captures non-linear interactions without demanding feature scaling. It was chosen as the final method for this submission; it is **not** claimed to be superior to methods that have not been compared here.

**Hyperparameters:** `n_estimators=250` averages 250 diverse trees for stable predictions at modest computational cost; `min_samples_leaf=5` prevents leaves based on very few users; `max_features='sqrt'` decorrelates trees; `class_weight='balanced_subsample'` adjusts for any training-class imbalance within bootstrap samples; and `random_state=42` makes results reproducible. `n_jobs=-1` uses available cores. There is **no learning rate, optimiser or epoch count** for Random Forest: it builds decision trees instead of performing gradient-descent training. A majority-class dummy baseline gives necessary context.

The imputers and encoder sit **inside a pipeline** that is fitted exclusively with training data. No test labels participate in model selection or training.

In [5]:
numeric_pipe = Pipeline([('impute', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocess = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features)
])
model = Pipeline([
    ('preprocess', preprocess),
    ('classifier', RandomForestClassifier(n_estimators=250, min_samples_leaf=5,
                                          max_features='sqrt', class_weight='balanced_subsample',
                                          random_state=RANDOM_STATE, n_jobs=-1))
])
model.fit(X_train, y_train)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print('Random Forest and majority-class baseline fitted using training data only.')

Random Forest and majority-class baseline fitted using training data only.
